# Taller de visualización de datos en R

![](https://ggplot2.tidyverse.org/logo.png)
![](https://tidyverse.org/css/images/hex/dplyr.png)

### **Profesor:** Fernando Salcedo Mejía, Eco Msc.
### Programa de Ciencias de Datos – 2026-1

Este cuaderno está diseñado para ejecutarse en **Google Colab con kernel R**.

> En Colab: *Runtime → Change runtime type → R*

## 1. Librerías

In [ ]:
install.packages(c("tidyverse", "shiny", "GGally"))

In [ ]:
library(tidyverse)
library(GGally)
library(shiny)

In [ ]:
# Tamaño estándar de gráficos para el notebook
set_plot_size <- function(width = 10, height = 7) {
  options(repr.plot.width = width, repr.plot.height = height)
}
# Aplicar
set_plot_size()

# Temas para todos los graficos
theme_set(
  theme_minimal(base_size = 20)
)

## 2. Carga de datos

In [ ]:
github_datos <- "https://raw.githubusercontent.com/fersalme/programacion-python-r/refs/heads/main/datos/palmerpenguins_extended.csv"

df_pinguinos <- read_csv(github_datos)
head(df_pinguinos)

## 3. Gráfico de barras: total por especie

In [ ]:
df_especies <- df_pinguinos %>%
  count(species, name = "recuento")

ggplot(df_especies, aes(x = species, y = recuento)) +
  geom_col(fill = "steelblue") +
  labs(title = "Recuento de especies de pingüinos",
       x = "Especie", y = "Total")

## 4. Tarea: promedio de peso por especie

In [ ]:
# TU CODIGO AQUI

## 5. Peso promedio por año

In [ ]:
df_peso_pinguinos <- df_pinguinos %>%
  group_by(year) %>%
  summarise(body_mass_g = mean(body_mass_g, na.rm = TRUE))

ggplot(df_peso_pinguinos, aes(x = year, y = body_mass_g)) +
  geom_line() + geom_point() +
  scale_x_continuous(breaks = df_peso_pinguinos$year) +
  labs(title = "Peso promedio de los pingüinos por año",
       x = "Años", y = "Peso (gr)")

## 6. Líneas por año y especie

In [ ]:
df_peso_year <- df_pinguinos %>%
  group_by(year, species) %>%
  summarise(body_mass_g = mean(body_mass_g, na.rm = TRUE))

ggplot(df_peso_year, aes(x = year, y = body_mass_g, color = species)) +
  geom_line() +
  geom_point() +
  scale_x_continuous(breaks = unique(df_peso_year$year)) +
  labs(
    title = "Peso promedio de los pingüinos por año y especie",
    x = "Años",
    y = "Peso (gr)",
    color = "Especie"
  )

## 7. Scatterplot Matrix

In [ ]:
df_pair <- df_pinguinos %>%
  select(bill_depth_mm, body_mass_g, bill_length_mm,
         flipper_length_mm, species)

ggpairs(df_pair, aes(color = species))

## 8. Histogramas

In [ ]:
  ggplot(df_pinguinos, aes(x = body_mass_g)) +
  geom_histogram(bins = 10, fill = "skyblue", color = "white") +
  labs(
    title = "Histograma peso de los pingüinos",
    x = "Peso (gr)",
    y = "Frecuencia"
  )


## 9. Densidad por especie

In [ ]:
ggplot(df_pinguinos, aes(x = body_mass_g, fill = species)) +
  geom_density(alpha = 0.5) +
  labs(title = "Densidad peso de los pingüinos por especies")

## 10. Boxplot

In [ ]:
ggplot(df_pinguinos, aes(x = species, y = body_mass_g, fill = species)) +
  geom_boxplot() +
  labs(
    title = "Peso de los pingüinos por especie",
    x = "Especie",
    y = "Peso (gr)"
  )

## 11. Heatmap de correlación

In [ ]:
df_num <- df_pinguinos %>%
  select(bill_depth_mm, body_mass_g, bill_length_mm, flipper_length_mm)

corr <- cor(df_num, use = "complete.obs")

corr_long <- as.data.frame(as.table(corr))

ggplot(corr_long, aes(Var1, Var2, fill = Freq)) +
  geom_tile() +
  geom_text(aes(label = round(Freq, 2)), color = "white") +
  scale_fill_gradient(low = "lightblue", high = "darkblue") +
  labs(title = "Correlación entre medidas de los pingüinos")

## 12. Scatterplot Matrix

In [ ]:
df_pair <- df_pinguinos %>%
  select(bill_depth_mm, body_mass_g, bill_length_mm, flipper_length_mm, species)

ggpairs(df_pair, aes(color = species))

## 13. Dashboard interactivo (Shiny)

In [ ]:
ui <- fluidPage(
  selectInput("especie", "Especie:", choices = unique(df_pinguinos$species)),
  plotOutput("grafico")
)

server <- function(input, output) {
  output$grafico <- renderPlot({
    df_pinguinos %>%
      filter(species == input$especie) %>%
      ggplot(aes(x = bill_depth_mm, y = body_mass_g)) +
      geom_point(color = "steelblue") +
      theme_minimal()
  })
}

# Ejecutar con:
# shinyApp(ui, server)